# 01 — Data Analysis: Hypergamy Engagement Collection Pipeline

**Author:** Yertis Ydyrys · **Course:** TASK 3 (W7–W9) · **Track:** ML / AI

This notebook processes the raw pipeline benchmark log (`data/raw/test_run.jsonl`):
it filters out the cold-start warm-up phase, computes
descriptive statistics (median, p25–p75, p95, p99), derives **bootstrap 95%
confidence intervals** for the mean latency, tracks the **guardrail error rate**,
and exports publication-ready figures to `reports/figures/`.

> **Data note.** The committed `test_run.jsonl` is a **simulated (synthetic) log** generated with a
> fixed seed by `src/generate_data.py`; it is *not* a real measurement. It lets the full pipeline run
> from a clean clone, so the numbers below demonstrate the analysis method rather than empirical
> findings. To analyse real collection logs, replace `data/raw/test_run.jsonl` (same schema) and
> re-run all cells.

## 1. Setup & reproducibility

All randomness (bootstrap resampling) is pinned to `seed = 42` so every run
produces identical results (`Kernel -> Restart & Run All` safe).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

np.random.seed(42)  # global seed, per rubric

# make src/ importable when the notebook is run from /notebooks
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from process_data import load_jsonl, filter_warmup, bootstrap_ci, summarize, RAW_PATH, OUT_PATH
from visualize import main as run_visualize

pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
print(f"Project root: {ROOT}")

Project root: /mnt/agents/output/hypergamy-task3


## 2. Data ingestion

Load the raw JSONL log. Every record is one run of the collection pipeline:
`run_id, phase (warmup|eval), method (baseline|proposed), platform,
latency_ms, throughput, error`.

In [2]:
raw = load_jsonl(RAW_PATH)
print(f"Raw records: {len(raw)}")
raw.groupby(["method", "platform", "phase"]).size().unstack()

Raw records: 972


phase                     eval  warmup
method   platform                     
baseline Instagram Reels   150      12
         TikTok            150      12
         YouTube Long      150      12
proposed Instagram Reels   150      12
         TikTok            150      12
         YouTube Long      150      12

## 3. Warm-up filtering

Cold-start runs (`phase == "warmup"`) show connection-pool warm-up spikes and
must be discarded before any statistics are computed.

In [3]:
eval_df = filter_warmup(raw)
print(f"Before filtering: {len(raw)} runs | after: {len(eval_df)} runs "
      f"({len(eval_df) / len(raw):.1%} kept)")

warm = raw[raw['phase'] == 'warmup']['latency_ms']
cold = eval_df['latency_ms']
print(f"Warm-up latency:  mean={warm.mean():.1f} ms (discarded)")
print(f"Eval latency:     mean={cold.mean():.1f} ms (analyzed)")

Before filtering: 972 runs | after: 900 runs (92.6% kept)
Warm-up latency:  mean=433.2 ms (discarded)
Eval latency:     mean=134.1 ms (analyzed)


## 4. Statistical aggregation

Per (method, platform) group we report: **Mean±SD**, **Median (p25–p75)**,
tail percentiles **p95 / p99**, a **percentile-bootstrap 95% CI** for the mean
latency (10,000 resamples, seed 42), mean throughput, and the **guardrail error
rate** (share of failed runs).

In [4]:
summary = summarize(eval_df)
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUT_PATH, index=False)
print(f"Saved -> {OUT_PATH}")
summary

Saved -> /mnt/agents/output/hypergamy-task3/data/processed/metrics_summary.csv


,method,platform,n_runs,latency_mean_ms,latency_std_ms,latency_median_ms,latency_p25_ms,latency_p75_ms,latency_p95_ms,latency_p99_ms,ci95_low_ms,ci95_high_ms,throughput_mean_rps,error_rate
0,baseline,Instagram Reels,150,146.214,26.386,148.025,128.947,160.668,191.797,204.460,142.044,150.442,37.351,0.007
1,baseline,TikTok,150,126.525,23.175,125.280,112.590,139.185,167.760,188.413,122.749,130.155,41.955,0.000
2,baseline,YouTube Long,150,187.452,32.201,187.240,164.905,207.575,241.828,251.543,182.318,192.586,30.083,0.020
3,proposed,Instagram Reels,150,106.587,19.414,108.025,95.638,118.135,139.024,154.287,103.451,109.647,55.523,0.000
4,proposed,TikTok,150,94.148,18.030,92.825,82.218,107.030,124.519,133.952,91.267,97.036,60.063,0.007
5,proposed,YouTube Long,150,143.572,25.677,142.945,125.670,159.290,189.017,205.452,139.420,147.836,44.619,0.020


### Sanity checks

- Error rate stays below the 5% guardrail threshold in every group.
- The proposed (concurrent) collector is consistently faster than the baseline.

In [5]:
guardrail_ok = (summary["error_rate"] < 0.05).all()
print(f"Guardrail error_rate < 5% in all groups: {guardrail_ok}")
summary.pivot(index="platform", columns="method", values="latency_mean_ms").round(1)

Guardrail error_rate < 5% in all groups: True


method,baseline,proposed
platform,,
Instagram Reels,146.200,106.600
TikTok,126.500,94.100
YouTube Long,187.500,143.600


## 5. Visualization & export

Both figures follow the three scientific-visualization rules: physical units on
every axis, visible 95% CI (error bars / boxplots), colorblind-safe palette with
distinct markers and line styles for grayscale readability. Exports: vector
`.pdf` + `.png` at 300 DPI in `reports/figures/`.

In [6]:
run_visualize()
sorted(Path(ROOT / "reports" / "figures").glob("*.*"))

[visualize] saved fig1_latency_vs_load.pdf / fig1_latency_vs_load.png


/mnt/agents/output/hypergamy-task3/src/visualize.py:77: UserWarning: The palette list has more values (10) than needed (2), which may not be intended.
  sns.boxplot(data=df, x="platform", y="latency_ms", hue="method", order=order,


[visualize] saved fig2_distribution_boxplot.pdf / fig2_distribution_boxplot.png


[PosixPath('/mnt/agents/output/hypergamy-task3/reports/figures/fig1_latency_vs_load.pdf'),
 PosixPath('/mnt/agents/output/hypergamy-task3/reports/figures/fig1_latency_vs_load.png'),
 PosixPath('/mnt/agents/output/hypergamy-task3/reports/figures/fig2_distribution_boxplot.pdf'),
 PosixPath('/mnt/agents/output/hypergamy-task3/reports/figures/fig2_distribution_boxplot.png')]